# Rétropropagation : suivre deux poids à la main

Deux poids suffisent pour comprendre la règle de la chaîne.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#retropropagation-a-la-main).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Une sortie dépend parfois de plusieurs poids successifs. Nous gardons deux multiplications pour suivre exactement chaque influence, sans ajouter de classe ou de réseau compliqué.

Les valeurs sont choisies : x=2, w1=3 et w2=−1. Nous obtenons h=6, y=−6 puis L=(y−0)²=36.

Les étiquettes des flèches décrivent le calcul direct. Le tableau suivant suit les influences en sens inverse.

| Quantité cherchée | Calcul | Valeur |
| --- | --- | --- |
| dL/dy | 2 × (−6 − 0) | −12 |
| dL/dw2 | dL/dy × h = −12 × 6 | −72 |
| dL/dh | dL/dy × w2 = −12 × (−1) | 12 |
| dL/dw1 | dL/dh × x = 12 × 2 | 24 |

Pour une multiplication, la dérivée par rapport à un facteur est l’autre facteur. Chaque chemin reprend ensuite le gradient reçu.

Pour une multiplication, la dérivée par rapport à un facteur est l’autre facteur. La règle de la chaîne multiplie ces influences le long du trajet.

Les poids w1 et w2 sont créés directement avec requires_grad=True. PyTorch conserve normalement leur grad. h et y résultent d’opérations : nous demandons retain_grad pour inspecter aussi leurs dérivées.

## Démonstration 1 · Lire les paramètres et les intermédiaires

Faire correspondre chaque facteur manuel à un gradient PyTorch.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

L’entrée est fixée. Les deux poids flottants demandent le suivi de leurs influences.

In [4]:
x = torch.tensor(2.)
w1 = torch.tensor(3., requires_grad=True)
w2 = torch.tensor(-1., requires_grad=True)


In [5]:
observer('x', x)
observer('w1', w1)
observer('w2', w2)

x forme () type torch.float32 requires_grad False
  valeurs : 2.0
w1 forme () type torch.float32 requires_grad True
  valeurs : 3.0
w2 forme () type torch.float32 requires_grad True
  valeurs : -1.0


### Calculer l’état suivant

h dépend de w1 par la multiplication par x. Ce calcul crée un intermédiaire de valeur 6.

In [6]:
h = w1 * x


In [7]:
observer('h', h)

h forme () type torch.float32 requires_grad True
  valeurs : 6.0


### Calculer l’état suivant

retain_grad ne modifie pas h. Il demande de conserver sa dérivée après backward, pour que nous puissions la lire.

Sans cet appel, les gradients des poids resteraient calculés. C’est l’observation de h.grad qui manquerait.

In [8]:
h.retain_grad()


### Calculer l’état suivant

La seconde multiplication donne y=−6. Nous demandons ensuite, avec la même méthode, de conserver sa dérivée.

In [9]:
y = w2 * h
y.retain_grad()


In [10]:
observer('y', y)

y forme () type torch.float32 requires_grad True
  valeurs : -6.0


### Calculer l’état suivant

Le carré de l’écart à zéro donne L=36. C’est de ce scalaire que nous allons repartir à rebours.

In [11]:
perte = (y - 0) ** 2


In [12]:
observer('perte', perte)

perte forme () type torch.float32 requires_grad True
  valeurs : 36.0


### Calculer les gradients

backward utilise le graphe construit pendant la passe directe. −12×6 donne −72 pour w2 ; −12×(−1)×2 donne 24 pour w1.

In [13]:
perte.backward()


In [14]:
observer_entrainement({'x': x, 'w1': w1, 'w2': w2, 'h': h, 'y': y, 'perte': perte}, montrer_poids=False)

w1.grad forme () type torch.float32 requires_grad False
  valeurs : 24.0
w2.grad forme () type torch.float32 requires_grad False
  valeurs : -72.0


### Lire les sorties

item() extrait les scalaires uniquement pour les afficher. Aucun poids n’a été modifié.

In [15]:
print("h, y, perte", h.item(), y.item(), perte.item())
print("gradients intermediaires", y.grad.item(), h.grad.item())
print("gradients poids", w1.grad.item(), w2.grad.item())

h, y, perte 6.0 -6.0 36.0
gradients intermediaires -12.0 12.0
gradients poids 24.0 -72.0


Nous retrouvons les mêmes produits que dans le tableau. retain_grad explique l’observation des intermédiaires ; backward suffit à calculer les gradients des poids.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Complétez le gradient du premier poids dans le graphe x=2, w1=3, w2=−1.

In [16]:
exo_w1 = torch.tensor(3.,requires_grad=True)
exo_w2 = torch.tensor(-1.,requires_grad=True)
exo_h = exo_w1*2
exo_y = exo_w2*exo_h
(exo_y**2).backward()
exo_gradient = ...  # Remplacer les points par votre réponse.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w1.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Remontez −12 vers h avec le facteur −1, puis vers w1 avec le facteur 2.

</details>

### Exercice 2

Complétez cette fois le gradient du second poids, avec le même calcul.

In [17]:
exo_w1 = torch.tensor(3.,requires_grad=True)
exo_w2 = torch.tensor(-1.,requires_grad=True)
exo_h = exo_w1*2
exo_y = exo_w2*exo_h
(exo_y**2).backward()
exo_gradient = ...  # Remplacer les points par votre réponse.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w2.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

La variation locale de y par rapport à w2 vaut h=6. Multipliez-la par −12.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_w1 = torch.tensor(3.,requires_grad=True)
exo_w2 = torch.tensor(-1.,requires_grad=True)
exo_h = exo_w1*2
exo_y = exo_w2*exo_h
(exo_y**2).backward()
exo_gradient = 24.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w1.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_w1 = torch.tensor(3.,requires_grad=True)
exo_w2 = torch.tensor(-1.,requires_grad=True)
exo_h = exo_w1*2
exo_y = exo_w2*exo_h
(exo_y**2).backward()
exo_gradient = -72.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w2.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [18]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()